In [1]:
!pip install scikeras
!pip install imbalanced-learn xgboost lightgbm catboost
!pip install scikit-learn==1.5.2 imbalanced-learn==0.12.3 scikeras==0.13.0 -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.9/12.9 MB 105.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 258.3/258.3 kB 19.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tpot 1.1.0 requires scikit-learn>=1.6, but you have scikit-learn 1.5.2 which is incompatible.
category-encoders 2.9.0 requires scikit-learn>=1.6.0, but you have scikit-learn 1.5.2 which is incompatible.
umap-learn 0.5.11 requires scikit-learn>=1.6, but you have scikit-learn 1.5.2 which is incompatible.
hdbscan 0.8.41 requires scikit-learn>=1.6, but you have scikit-learn 1.5.2 which is incompatible.


In [2]:
# ==============================================================================
# 🗺️ ESTIMASI PRODUKSI PADI MENGGUNAKAN DATA CITRA RADAR & OPTIK
# MODEL: LSTM / BI-LSTM + DEEP FUSION TUNING (T4x2 GPU)
# DATASET: unik_per_bulan.csv (Proporsi Non-Padi Imbang)
# ==============================================================================

import pandas as pd
import numpy as np
import os, gc, time, joblib, warnings
import matplotlib.pyplot as plt
import seaborn as sns

# TensorFlow & SciKeras
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Dense, Reshape, Input, Dropout, Concatenate, 
                                     Lambda, BatchNormalization)
from tensorflow.keras.layers import LSTM as KerasLSTM
from tensorflow.keras.layers import Bidirectional
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from scikeras.wrappers import KerasClassifier

# Preprocessing & Metrics
from sklearn.model_selection import train_test_split, RandomizedSearchCV, TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (accuracy_score, f1_score, cohen_kappa_score,
                             precision_score, recall_score, confusion_matrix)

warnings.filterwarnings('ignore')

# ==========================================================
# 🎯 0. CONTROL PANEL (GANTI MODEL DI SINI)
# ==========================================================
MODEL_PILIHAN = 'BiLSTM'   # Pilihan: 'LSTM' atau 'BiLSTM'

print(f"🔥 MEMULAI NOTEBOOK UNTUK: {MODEL_PILIHAN.upper()} 🔥\n")

# ==========================================================
# 🔍 1. PEMUATAN DATASET LANGSUNG DARI PATH
# ==========================================================
dataset_path = '/kaggle/input/datasets/jejehayu/data-klasifikasi-padi/unik_per_bulan.csv'
print(f"🔍 Membaca dataset dari: {dataset_path}")

if os.path.exists(dataset_path):
    # Penyesuaian kolom berdasarkan format data terbaru (LSWI dan EVI bertukar posisi)
    kolom_input = ['id', 'longitude', 'latitude', 'Tahun', 'Bulan', 'date_img', 'Lahan',
                   'VV', 'VH', 'RVI', 'NDVI', 'LSWI', 'EVI', 'BSI']
    
    # Header=0 digunakan karena file csv memiliki nama kolom asli di baris pertama
    df = pd.read_csv(dataset_path, names=kolom_input, header=0)
    df['date_img'] = pd.to_datetime(df['date_img'])

    # Eksklusi data tahun 2021 jika ada
    df = df[df['Tahun'] != 2021].reset_index(drop=True)
    print(f"✅ Dataset berhasil dimuat! Jumlah baris: {len(df)}")
else:
    raise FileNotFoundError(f"❌ ERROR: File {dataset_path} tidak ditemukan.")

# ==========================================================
# ⚙️ 2. FEATURE ENGINEERING (WAKTU & LAG SATELIT)
# ==========================================================
df['segmen'] = df['longitude'].astype(str) + '_' + df['latitude'].astype(str)
df = df.sort_values(['segmen', 'Tahun', 'Bulan']).reset_index(drop=True)

df['Bulan_sin'] = np.sin(2 * np.pi * df['Bulan'] / 12)
df['Bulan_cos'] = np.cos(2 * np.pi * df['Bulan'] / 12)

# Sesuai dengan urutan kolom data terbaru
features_sat = ['VV', 'VH', 'RVI', 'NDVI', 'LSWI', 'EVI', 'BSI']

for lag in [1, 2, 3]:
    for feat in features_sat:
        df[f'{feat}_lag{lag}'] = df.groupby('segmen')[feat].shift(lag)

df_panel = df.dropna(subset=['Lahan']).reset_index(drop=True)

# ==========================================================
# ⚖️ 3. BALANCING DATA: PADI VS BUKAN PADI (PROPORSI IMBANG)
# ==========================================================
print("⚖️ Melakukan penyeimbangan data dinamis...")

# Standardisasi teks Lahan
df_panel['Lahan'] = df_panel['Lahan'].astype(str).str.strip().str.lower()

# Pisahkan kelas Padi
df_padi = df_panel[df_panel['Lahan'] == 'padi']
n_padi = len(df_padi)

# Temukan semua kelas selain Padi (Misal: bangunan, air, tanaman lain, dll)
kelas_non_padi = df_panel[df_panel['Lahan'] != 'padi']['Lahan'].unique()
k_classes = len(kelas_non_padi)

if k_classes > 0:
    print(f"📌 Ditemukan {k_classes} kelas Non-Padi: {kelas_non_padi}")
    
    # Hitung jatah ideal untuk tiap kelas non-padi agar totalnya == n_padi
    kuota_per_kelas = n_padi // k_classes
    sisa = n_padi % k_classes
    
    df_non_padi_terpilih = []
    
    for i, cls in enumerate(kelas_non_padi):
        df_cls = df_panel[df_panel['Lahan'] == cls]
        # Distribusikan sisa kuota ke kelas-kelas pertama
        target_n = kuota_per_kelas + (1 if i < sisa else 0)
        
        # Ambil sampel (jika jumlah asli kurang dari target, ambil semua)
        n_ambil = min(target_n, len(df_cls))
        df_non_padi_terpilih.append(df_cls.sample(n=n_ambil, random_state=42))
        
    df_non_padi_balanced = pd.concat(df_non_padi_terpilih)
    df_balanced = pd.concat([df_padi, df_non_padi_balanced])
else:
    # Fallback jika ternyata hanya ada padi di dataset
    df_balanced = df_padi

# Beri label biner (1 = Padi, 0 = Bukan Padi)
df_balanced['Label_Biner'] = np.where(df_balanced['Lahan'] == 'padi', 1, 0)

print("\n📊 RINGKASAN DATA SETELAH BALANCING:")
print(f"Total Padi (1)      : {sum(df_balanced['Label_Biner'] == 1)}")
print(f"Total Bukan Padi (0): {sum(df_balanced['Label_Biner'] == 0)}")
print("Rincian kelas asli pada Bukan Padi:")
print(df_balanced[df_balanced['Label_Biner'] == 0]['Lahan'].value_counts())
print("\n")

df_panel = df_balanced.sort_values(['Tahun', 'Bulan', 'segmen']).reset_index(drop=True)

# Menyusun kolom input model
kolom_temporal = []
for lag in [3, 2, 1]:
    for feat in features_sat:
        kolom_temporal.append(f'{feat}_lag{lag}')
for feat in features_sat:
    kolom_temporal.append(feat)

spasial = ['Bulan_sin', 'Bulan_cos', 'Tahun']
feats_all = kolom_temporal + spasial  # Total 31 fitur input

X = df_panel[feats_all].astype(np.float32)
y = df_panel['Label_Biner']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

# ==========================================================
# 🧠 4. DEFINISI ARSITEKTUR MODEL (DEEP FUSION & T4x2)
# ==========================================================
N_SAT_FEATURES = len(features_sat) # 7
N_TIMESTEP = 4
N_TEMPORAL_COLS = N_SAT_FEATURES * N_TIMESTEP 

# 1. INISIALISASI STRATEGI MULTI-GPU SEKALI SAJA (GLOBAL)
strategy = tf.distribute.MirroredStrategy()
print(f"🚀 Jumlah GPU yang tersinkronisasi: {strategy.num_replicas_in_sync}")

def build_rnn_deep_fusion(meta, dropout_rate=0.2, learning_rate=1e-3, 
                          bidirectional=False, dense_1_units=64, dense_2_units=32):
    
    # 2. BERSIHKAN MEMORI GPU SEBELUM MEMBUAT MODEL BARU
    tf.keras.backend.clear_session()
    
    # 3. GUNAKAN SCOPE DARI STRATEGI GLOBAL
    with strategy.scope():
        n_features = meta["n_features_in_"]
        inputs = Input(shape=(n_features,), name='input_gabungan')
        
        x_sat = Lambda(lambda z: z[:, :N_TEMPORAL_COLS], name='slice_satelit')(inputs)
        x_time = Lambda(lambda z: z[:, N_TEMPORAL_COLS:], name='slice_waktu')(inputs)
        x_sat_reshaped = Reshape((N_TIMESTEP, N_SAT_FEATURES), name='reshape_3d')(x_sat)

        # STRUKTUR LSTM (122 -> 61)
        if bidirectional:
            rnn_1 = Bidirectional(KerasLSTM(122, return_sequences=True), name='bilstm_layer_1')(x_sat_reshaped)
            rnn_2 = Bidirectional(KerasLSTM(61, return_sequences=False), name='bilstm_layer_2')(rnn_1)
        else:
            rnn_1 = KerasLSTM(122, return_sequences=True, name='lstm_layer_1')(x_sat_reshaped)
            rnn_2 = KerasLSTM(61, return_sequences=False, name='lstm_layer_2')(rnn_1)

        norm_layer = BatchNormalization(name='batch_norm_1')(rnn_2)
        drop_layer = Dropout(dropout_rate, name='dropout_1')(norm_layer)
        
        merged = Concatenate(name='gabung_fitur')([drop_layer, x_time])
        
        # EKSTRA DENSE LAYER (DEEP FUSION)
        dense_1 = Dense(dense_1_units, activation='relu', name='dense_fusion_1')(merged)
        dense_1 = BatchNormalization(name='batch_norm_fusion_1')(dense_1)
        
        dense_2 = Dense(dense_2_units, activation='relu', name='dense_fusion_2')(dense_1)
        dense_2 = BatchNormalization(name='batch_norm_fusion_2')(dense_2)
        
        outputs = Dense(1, activation='sigmoid', name='output')(dense_2)

        model = Model(inputs=inputs, outputs=outputs)
        model.compile(optimizer=Adam(learning_rate=learning_rate), loss='binary_crossentropy', metrics=['accuracy'])
        
    return model

# ==========================================================
# 🚀 5. TRAINING & HYPERPARAMETER TUNING
# ==========================================================
is_bidirectional = (MODEL_PILIHAN.upper() == 'BILSTM')
early_stop = EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True, verbose=0)

rnn_model = KerasClassifier(
    model=build_rnn_deep_fusion,
    dropout_rate=0.2,
    learning_rate=1e-3,
    bidirectional=is_bidirectional,
    dense_1_units=64,
    dense_2_units=32,
    epochs=40,
    batch_size=256,
    validation_split=0.15,
    callbacks=[early_stop],
    verbose=0,
)

pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('model', rnn_model)
])

tscv = TimeSeriesSplit(n_splits=2)

param_grid = {
    'model__dropout_rate': [0.1, 0.2, 0.3],
    'model__learning_rate': [1e-3, 5e-4],
    'model__batch_size': [128, 256, 512],
    'model__dense_1_units': [32, 64, 128],
    'model__dense_2_units': [16, 32, 64],
}

tuner = RandomizedSearchCV(
    pipe, param_grid, cv=tscv, scoring='f1_macro',
    n_iter=10, n_jobs=1, random_state=42, verbose=1
)

print(f"🚀 Memulai Training {MODEL_PILIHAN} Deep Fusion (T4x2 Distributed)...")
start_time = time.time()
tuner.fit(X_train, y_train)

# ==========================================================
# 📊 6. EVALUASI & PENYIMPANAN
# ==========================================================
best_model = tuner.best_estimator_
y_pred = best_model.predict(X_test)

acc = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average='macro')
kappa = cohen_kappa_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, average='macro')
rec = recall_score(y_test, y_pred, average='macro')
waktu = time.time() - start_time

print(f"\n✅ TRAINING {MODEL_PILIHAN} SELESAI!")
print(f"📊 F1-Macro: {f1:.4f} | Acc: {acc:.4f} | Kappa: {kappa:.4f}")
print(f"⏱️ Waktu: {waktu:.1f} detik")
print(f"🔧 Parameter Terbaik: {tuner.best_params_}")

nama_bersih = f"{MODEL_PILIHAN}_DeepFusion".replace(" ", "")

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Oranges')
plt.title(f'Confusion Matrix - {MODEL_PILIHAN} (Deep Fusion)')
plt.ylabel('Aktual')
plt.xlabel('Prediksi')
plt.savefig(f'/kaggle/working/CM_{nama_bersih}.png')
plt.close()

file_hasil = f'/kaggle/working/Summary_Models.csv'
log_data = {
    'Model': f"{MODEL_PILIHAN} (Deep Fusion)",
    'Accuracy': acc,
    'F1_Macro': f1,
    'Kappa': kappa,
    'Precision_Macro': prec,
    'Recall_Macro': rec,
    'Waktu_Sekon': round(waktu, 2),
    'Best_Params': str(tuner.best_params_)
}

if os.path.exists(file_hasil):
    df_log = pd.read_csv(file_hasil)
    df_log = pd.concat([df_log, pd.DataFrame([log_data])], ignore_index=True)
else:
    df_log = pd.DataFrame([log_data])
df_log.to_csv(file_hasil, index=False)

joblib.dump(best_model, f'/kaggle/working/Model_{nama_bersih}_Final.pkl')
gc.collect()

print(f"📂 Semua file telah disimpan di folder output Kaggle!")

2026-09-12 15:07:08.661288: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1789225628.844658      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1789225628.900830      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1789225629.354012      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1789225629.354048      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1789225629.354051      23 computation_placer.cc:177] computation placer alr

🔥 MEMULAI NOTEBOOK UNTUK: BILSTM 🔥

🔍 Membaca dataset dari: /kaggle/input/datasets/jejehayu/data-klasifikasi-padi/unik_per_bulan.csv
✅ Dataset berhasil dimuat! Jumlah baris: 151074
⚖️ Melakukan penyeimbangan data dinamis...
📌 Ditemukan 5 kelas Non-Padi: ['bangunan' 'hutan' 'puso' 'tanaman lain' 'air']

📊 RINGKASAN DATA SETELAH BALANCING:
Total Padi (1)      : 50835
Total Bukan Padi (0): 32178
Rincian kelas asli pada Bukan Padi:
Lahan
bangunan        10167
hutan           10167
air             10167
puso              860
tanaman lain      817
Name: count, dtype: int64


INFO:tensorflow:Using MirroredStrategy with devices ('/job:localhost/replica:0/task:0/device:GPU:0', '/job:localhost/replica:0/task:0/device:GPU:1')


I0000 00:00:1789225654.310752      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1789225654.316604      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


🚀 Jumlah GPU yang tersinkronisasi: 2
🚀 Memulai Training BiLSTM Deep Fusion (T4x2 Distributed)...
Fitting 2 folds for each of 10 candidates, totalling 20 fits
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
IN

I0000 00:00:1789225662.327131      88 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1789225663.190730      85 cuda_dnn.cc:529] Loaded cuDNN version 91002


INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Collective all_reduce tensors: 24 all_reduces, num_devices = 2, group_size = 2, implementation = CommunicationImplementation.NCCL, num_packs = 1
INFO:tensorflow:Collective all_reduce tensors: 24 all_reduces, num_devices = 2, group_size = 2, implementation = CommunicationImplementation.NCCL, num_packs = 1
INFO:tensorflow:Collective all_reduce tensors: 24 all_reduces, num_devices = 2, group_size = 2, implementation = CommunicationImplementation.NCCL, num_packs = 1
INFO:tensorflow:Collective all_reduce tensors: 24 all_reduces, num_devices = 2, group_size = 2, implementation = CommunicationImplementation.NCCL, num_packs = 1
INFO:tensorflow:Collective all_reduce tensors: 24 all_reduces, num_devices =